In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras import layers, models
from PIL import ImageFile
import pandas as pd
import numpy as np
import os

ImageFile.LOAD_TRUNCATED_IMAGES = True
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [ ]:
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential()
model.add(base_model)
model.add(layers.GlobalMaxPooling2D())
model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 512)            │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# train + vali

In [ ]:
trainValiPath='/content/gdrive/MyDrive/ml/train+vali'
trainValiImage = datagen.flow_from_directory(trainValiPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)
trainValifeatures = model.predict(trainValiImage, verbose=1)

Found 2954 images belonging to 18 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


93/93 ━━━━━━━━━━━━━━━━━━━━ 961s 10s/step


In [ ]:
trainValifilenames = trainValiImage.filenames
trainValifeatures = trainValifeatures / np.linalg.norm(trainValifeatures, axis=1, keepdims=True)
df_trainVali = pd.DataFrame(trainValifeatures)
df_trainVali.insert(0, "filename", trainValifilenames)
df_trainVali.to_csv("trainValiFeature.csv", index=False)

# seperate train and vali

In [ ]:
trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

In [ ]:
trainImage = datagen.flow_from_directory(trainPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False )
valiImage= datagen.flow_from_directory(valiPath, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False )

Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


In [ ]:
trainLabel = []
valiLabel = []

for folder in sorted(os.listdir(trainPath)):
    folder_path = os.path.join(trainPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                trainLabel.append((label, label+"/"+file))

for folder in sorted(os.listdir(valiPath)):
    folder_path = os.path.join(valiPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                valiLabel.append((label, label+"/"+file))


In [ ]:
train = pd.DataFrame(trainLabel, columns=['label', 'pathname'])
vali = pd.DataFrame(valiLabel, columns=['label', 'pathname'])

train.to_csv('trainLabel.csv', index=False)
vali.to_csv('valiLabel.csv', index=False)

In [ ]:
train_features = model.predict(trainImage, verbose=1)
val_features = model.predict(valiImage, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 746s 8s/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 200s 9s/step


In [ ]:
train_filenames = trainImage.filenames
val_filenames = valiImage.filenames

train_features = train_features / np.linalg.norm(train_features, axis=1, keepdims=True)
val_features = val_features / np.linalg.norm(val_features, axis=1, keepdims=True)

df_train = pd.DataFrame(train_features)
df_val = pd.DataFrame(val_features)

df_train.insert(0, "filename", train_filenames)
df_val.insert(0, "filename", val_filenames)

df_train.to_csv("trainFeature.csv", index=False)
df_val.to_csv("valiFeature.csv", index=False)

In [ ]:
import joblib

filename = 'vggFinetune.sav'
joblib.dump(model, filename)

['vggFinetune.sav']

In [ ]:
import shutil
shutil.move('trainValiFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('vggFinetune.sav', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('trainFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('trainLabel.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('valiLabel.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')
shutil.move('valiFeature.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/valiFeature.csv'

# End of VGG

In [ ]:
from google.colab import driv
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
trainFeature= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainFeature.csv')
valiFeature=pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/valiFeature.csv')

trainFilename=trainFeature['filename']
valiFilename=valiFeature['filename']

trainFeature=trainFeature.drop(['filename'],axis=1)
valiFeature=valiFeature.drop(['filename'],axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler
stdscaler = StandardScaler()
stdscaler.fit(trainFeature)

trainFeature = stdscaler.transform(trainFeature)
valiFeature = stdscaler.transform(valiFeature)

In [ ]:
trainLabel= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainLabel.csv')
valiLabel = pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/valiLabel.csv')

trainLabel=trainLabel.drop(['pathname'],axis=1)
valiLabel=valiLabel.drop(['pathname'],axis=1)

In [ ]:
trainLabel=pd.get_dummies(trainLabel['label'])
valiLabel=pd.get_dummies(valiLabel['label'])
valiLabel=valiLabel.reindex(columns=trainLabel.columns, fill_value=0)

trainLabel = trainLabel.astype(int)
valiLabel = valiLabel.astype(int)

In [ ]:
from tensorflow.keras.optimizers import Adam

In [ ]:
model = models.Sequential()
model.add(layers.Dense(512, input_shape=(512,)))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(512))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(18, activation='softmax'))
model.summary()

adamm = Adam(learning_rate=0.0001)
model.compile(optimizer=adamm, loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 18)             │         9,234 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 538,642 (2.05 MB)

 Trainable params: 536,594 (2.05 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    restore_best_weights=True
)

In [ ]:
history = model.fit(trainFeature, trainLabel, epochs=300,
                    validation_data=(valiFeature, valiLabel),
                    callbacks=[early_stop])

Epoch 1/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.1884 - loss: 2.7963 - val_accuracy: 0.6452 - val_loss: 1.5153
Epoch 2/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6960 - loss: 1.2887 - val_accuracy: 0.8436 - val_loss: 0.8141
Epoch 3/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8440 - loss: 0.7609 - val_accuracy: 0.9028 - val_loss: 0.5184
Epoch 4/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8873 - loss: 0.5618 - val_accuracy: 0.9317 - val_loss: 0.3685
Epoch 5/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9110 - loss: 0.4455 - val_accuracy: 0.9461 - val_loss: 0.2777
Epoch 6/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9345 - loss: 0.3367 - val_accuracy: 0.9632 - val_loss: 0.2138
Epoch 7/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9629 - loss: 0.2583 - val_accuracy: 0.9763 - val_loss: 0.1674
Epoch 8/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9671 - loss: 0.2219 - val_accuracy: 0.9816 - 

In [ ]:
new_model = models.Sequential(model.layers[:-1])

for old_layer, new_layer in zip(model.layers[:-1], new_model.layers):
    new_layer.set_weights(old_layer.get_weights())
new_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 512)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529,408 (2.02 MB)

 Trainable params: 527,360 (2.01 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [ ]:
joblib.dump(new_model, 'cnnVGGFinetune.sav')

['cnnVGGFinetune.sav']

# Train done

In [3]:
import joblib
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [8]:
stdscaler = joblib.load('/content/gdrive/MyDrive/ml/vggFinetune/stdScaler.pkl')
model = joblib.load('/content/gdrive/MyDrive/ml/vggFinetune/cnnVGGFinetune.sav')

In [4]:
trainValiFeature= pd.read_csv('/content/gdrive/MyDrive/ml/vggFinetune/trainValiFeature.csv')
trainValiFilename=trainValiFeature['filename']
trainValiFeature=trainValiFeature.drop(['filename'],axis=1)

In [ ]:
VGG16 = joblib.load('/content/gdrive/MyDrive/ml/vggFinetune/vggFinetune.sav')

In [9]:
trainValiFeatures = stdscaler.transform(trainValiFeature)
trainVali = model.predict(trainValiFeatures, verbose=1)

93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [12]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(trainVali)

trainVali = scaler.transform(trainVali)

In [14]:
df_trainVali = pd.DataFrame(trainVali)
df_trainVali.insert(0, "filename", trainValiFilename)

In [16]:
joblib.dump(scaler, 'minMaxScaler.pkl')

['minMaxScaler.pkl']

In [15]:
import joblib
df_train.to_csv("trainValiVectors.csv", index=False)
joblib.dump(scaler, 'minMaxScaler.pkl')
# joblib.dump(new_model, 'cnnVGGFinetune.sav')
joblib.dump(stdscaler, 'stdScaler.pkl')

NameError: name 'df_train' is not defined

In [ ]:
shutil.move('trainValiVectors.csv', '/content/gdrive/MyDrive/ml/vggFinetune/')

'/content/gdrive/MyDrive/ml/vggFinetune/trainValiVectors.csv'